# Stage 1b — IPF reconciliation

Reconciles the demographic MRP per-cell vehicle-type probabilities (rows) with the
Experian per-PUMA vehicle-type marginals (columns) using Iterative Proportional
Fitting, producing the **joint demographic–vehicle poststratification table** used
by the Stage-2 model.

- **Input:** `data/interim/new_post.csv`, `data/interim/new2.csv`
- **Output:** `data/interim/joint_poststrat_table.csv`

This notebook mirrors `ipf_reconciliation.py`; run either one.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

INTERIM = Path('..') / 'data' / 'interim'
N_PUMAS, N_TYPES, MAX_ITERS, TOL = 111, 12, 10, 1e-4
TYPE_COLS = [f'col_{j}' for j in range(1, N_TYPES + 1)]
MARG_COLS = [f'type{j}' for j in range(1, N_TYPES + 1)]

In [ ]:
def ipf_update(M, u, v):
    """One IPF sweep: scale rows to u, then columns to v."""
    r = M.sum(axis=1, keepdims=True)
    N = M * (u[:, None] / np.where(r == 0, 1, r))
    c = N.sum(axis=0, keepdims=True)
    O = N * (v[None, :] / np.where(c == 0, 1, c))
    d_u = np.linalg.norm(u - O.sum(axis=1), 2)
    d_v = np.linalg.norm(v - O.sum(axis=0), 2)
    return O, d_u, d_v

In [ ]:
cells   = pd.read_csv(INTERIM / 'new_post.csv')
margins = pd.read_csv(INTERIM / 'new2.csv')

frames = []
for i in range(1, N_PUMAS + 1):
    rows  = cells[cells['pums_idx'] == i]
    X     = rows[TYPE_COLS].to_numpy(float) * rows['perwt'].to_numpy(float)[:, None]
    u     = X.sum(axis=1)
    HH    = u.sum()
    v     = HH * margins.loc[margins['pums'] == i, MARG_COLS].to_numpy(float).ravel()
    M = X.copy()
    for _ in range(MAX_ITERS):
        M, d_u, d_v = ipf_update(M, u, v)
        if d_u <= TOL and d_v <= TOL:
            break
    df_i = pd.DataFrame(M, columns=TYPE_COLS); df_i['i'] = i
    frames.append(df_i)
    print(f'PUMA {i:3d}:  d_u = {d_u:.5f}, d_v = {d_v:.5f}')

In [ ]:
out = pd.concat(frames, ignore_index=True)
out.to_csv(INTERIM / 'joint_poststrat_table.csv', index=False)
out.head()